# JouvenceKB Exploration
Ce notebook permet d'inspecter et valider directement les données ingérées via LaminDB et en Parquet sur à `gs://jouvencekb/kg/v2/`.

In [ ]:
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.expanduser('~/.config/gcloud/application_default_credentials.json')

import pandas as pd
import lamindb as ln
import bionty as bt
import lnschema_pertdb as pt

from manage_db.kg_storage import open_kg_root, read_nodes, read_edges
from manage_db.kg_loader import KGLoader

# 1. Vérification de l'instance LaminDB
ln.connect("jkobject/jouvencekb")
print(ln.info())

### Bilan LaminDB

In [ ]:
# Vérification des entités d'ontologie bionty
print(f"Maladies (MONDO) in DB: {bt.Disease.filter().count()}")
print(f"Proteines/Genes (Ensembl) in DB: {bt.Gene.filter().count()}")
print(f"Drugs (Compound) in DB: {pt.Compound.filter().count()}")

# Vérification d'une stat
pt.Compound.filter(name="aspirin").df()

### Graphe complet (KGLoader / GCS Parquet)

In [ ]:
# 2. Chargement du graphe consolidé vers pandas/dask
kg = KGLoader("gs://jouvencekb/kg/v2")

print(f"Noeuds existants: {kg.node_types}")
print(f"Arêtes existantes: {list(kg.relations.keys())[:5]}... (+{len(kg.relations)-5})")

display(kg.summary())

In [ ]:
# Inspecter le format de l'arête cible 'molecule_targets_protein'
df_targets = read_edges(kg.root, "molecule_targets_protein")
df_targets.head()

In [ ]:
# Inspecter la liste des edges 'paper_mentions_disease'
df_lit = read_edges(kg.root, "paper_mentions_disease")
display(df_lit.head())
print("Top sources contributing to literature_mentions:")
df_lit['source'].value_counts()